# Uncertainty / sensitivity analysis (NTM)

Two lightweight analyses for the reviewer response:

1. **Measurement-noise floor** (aleatoric): how noisy is the torque data itself? Establishes the floor the model's RMSE is judged against.
2. **Input-noise sensitivity** (robustness): how much does the trained NTM degrade under sensor noise on the inputs? No retraining.

Both are thin wrappers around `current_setpoints.utils.uncertainty`.

In [1]:
import os
import sys

import torch

root_dir = os.path.abspath("..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.parameters import Flux_IEEEMachine2, IEEEMachine2
from current_setpoints.optimization import ModelAnalytical
from current_setpoints.utils import (
    build_residual_test_set,
    input_noise_sensitivity,
    load_aggregated_csv_data,
    load_neural_model,
    measurement_noise_floor,
)

DATA_DIR = os.path.join(root_dir, "data")
MAT_DIR = os.path.join(DATA_DIR, "250416_mereni")
CSV_PATH = os.path.join(DATA_DIR, "aggregated_file_means.csv")

# Article model: SiLU, test RMSE 0.034 -> the _0344 weight/scaler pair.
WEIGHTS = os.path.join(root_dir, "weights", "NTM_Weights_0344.pth")
SCALER = os.path.join(root_dir, "weights", "NTM_Scaler_0344.npy")
HIDDEN_SIZE, INPUT_SIZE = 12, 5
DEVICE = torch.device("cpu")

## 1. Measurement-noise floor (aleatoric)

Reads the raw per-operating-point `.mat` files, takes the torque time-series in each, and reports the within-point scatter (std) and the precision of the averaged target (std/√N). Read-only — nothing is saved.

In [2]:
floor = measurement_noise_floor(MAT_DIR)

Found 174 measurement files in c:\Users\matas\Desktop\CurrentSetpoints_main\data\250416_mereni
Processed 174 points  (skipped 0)
Samples per point: median=719, min=431, max=1726

--- Instantaneous measurement scatter (within-point torque std, Nm) ---
  median : 0.4029
  mean   : 0.4137
  IQR    : [0.2290, 0.5831]
  range  : [0.0824, 0.9266]

--- Precision of the averaged per-point target (SEM = std/sqrt(N), Nm) ---
    (i.i.d. lower bound; true value is larger because ripple is autocorrelated)
  median : 0.0136
  IQR    : [0.0099, 0.0179]


## 2. Input-noise sensitivity (robustness)

Rebuild the exact residual-target test split the NTM was trained on, load the trained network, then inject 2% and 5% per-channel Gaussian noise into the test inputs and measure the RMSE degradation (averaged over many draws, no retraining).

In [3]:
COL_MAP = {c: c for c in ["omega", "id1", "iq1", "id3", "iq3", "torq"]}
df = load_aggregated_csv_data(CSV_PATH, COL_MAP)
X = df[["omega", "id1", "iq1", "id3", "iq3"]].values
y_meas = df[["torq"]].values

machine = IEEEMachine2()
machine.set_max_pars(curr_max=30.0, volt_max=13.0, omega_max=1800)
flux = Flux_IEEEMachine2()
analytical = ModelAnalytical(machine=machine, flux=flux)

# Residual target + fixed split (test_size=0.15, random_state=42 -> matches training).
X_test, y_test = build_residual_test_set(X, y_meas, analytical)

net, scaler = load_neural_model(
    WEIGHTS, SCALER, hidden_size=HIDDEN_SIZE, input_size=INPUT_SIZE, device=DEVICE
)

Loaded 174 valid data points from CSV.


In [4]:
sensitivity = input_noise_sensitivity(
    net, scaler, X_test, y_test, noise_levels=(0.0, 0.02, 0.05), n_trials=200
)

Test set: 27 points  |  200 noise realisations per level

   noise |   RMSE [Nm] |  vs clean
----------------------------------
     0% |      0.0344 |   100.0%
     2% |      0.0347 |   100.9%
     5% |      0.0361 |   105.0%


**Reading the results.** The clean (0%) RMSE should reproduce the reported test RMSE (≈0.034 Nm). It sits far below the instantaneous measurement scatter from part 1, and rises only marginally under 2–5% input noise — i.e. the model is not fitting measurement noise and is robust to sensor error.